# 第 3 周练习 —— HR 合成数据生成器（本地 Llama + Gradio）

## 练习目标（理念）

用开源指令模型 **meta-llama/Llama-3.2-3B-Instruct**，为业务场景（例如测试 HR 系统）搭建一个**合成数据生成器**：

1. 用 **BitsAndBytesConfig** 做 **4-bit NF4 量化**，降低显存占用，便于在 Colab 里跑
2. 写函数：让模型生成**结构化**员工记录，再解析成 **Pandas DataFrame**
3. （延伸）可用 Gradio / Google Sheets 把结果展示、导出

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 本地 / 开源 LLM | `AutoModelForCausalLM` + Hugging Face `MODEL_ID` |
| 量化（Quantization） | `BitsAndBytesConfig`：`load_in_4bit` + `nf4` |
| 合成数据（Synthetic Data） | prompt 约定 JSON schema，再 `json.loads` → DataFrame |
| Chat 模板 | `tokenizer.apply_chat_template(...)` |

## 怎么跑

1. 建议在 **Google Colab**（或同等 GPU 环境）从上到下运行
2. 准备好 Hugging Face Token（本笔记本用 `userdata.get('HF_TOKEN')`）
3. 先跑量化加载格，再跑生成函数测试格


## 定义数据策略

### 子任务

先想清楚合成数据生成器的**数据架构、输出格式、业务目的**——再写代码。这样 prompt、schema、下游 DataFrame 列名才能对齐。


### 综合数据策略：HR 系统测试

#### 1. 商业目的

主要目标：生成用于测试人力资源管理系统（HRIS）的高质量合成 HR 数据。这样可以做性能测试、UI 开发和分析原型，而**不必暴露**敏感的个人身份信息（PII）。

#### 2. 数据架构（员工记录）

每条生成记录代表一名「员工」，字段约定如下：

- **员工 ID**：唯一标识符（例如 `EMP-001`）
- **全名**：看起来真实的姓名
- **部门**：Engineering / Sales / Marketing / HR / Finance / Legal 之一
- **职位**：与部门匹配的职位（例如 Software Engineer、Account Manager）
- **薪资**：合理数值区间（约 $40,000 – $200,000）
- **雇用日期**：约 2010 年至今
- **绩效评级**：类别值（1–5，或 Excellent / Good 等）

#### 3. 输出格式

为兼容下游分析，模型应以 **JSON**（或 CSV）输出，便于无缝加载进 `pandas.DataFrame`。

#### 4. 约束和分布

- **实际薪资**：薪资应与职位、资历大致相关
- **部门平衡**：除非刻意偏置，记录应跨部门分布
- **唯一性**：员工 ID 在数据集中必须唯一


## 设置量化模型

使用 **BitsAndBytes** 的 **4-bit NF4** 量化初始化 `Llama-3.2-3B-Instruct`，以便在 Colab 等有限显存环境里高效加载与推理。


In [ ]:
# ========== 安装依赖：升级 bitsandbytes（量化运行时） ==========
# 感叹号 ! 是 notebook shell magic：在系统 shell 里执行，不是 Python 语句
# -U：升级到满足版本约束的较新包；>=0.46.1 是版本下界（字符串保持原样）
!uv install -U bitsandbytes>=0.46.1


In [ ]:
# ========== 导入 + 量化配置 + 登录 HF + 加载分词器与模型 ==========

# 从 transformers 导入：分词器、因果语言模型、BitsAndBytes 量化配置类
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# 导入 PyTorch：后面用 torch.bfloat16 作为计算/权重 dtype
import torch
# 从 Google Colab 导入 userdata：安全读取笔记本密钥（Secrets），避免写死在代码里
from google.colab import userdata
# 从 huggingface_hub 导入 login：用 Token 登录，才能拉受控模型权重
from huggingface_hub import login

# 1. 定义模型 ID（Hugging Face repo id；字符串必须与 Hub 上一致，不要改）
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# 2. 配置 BitsAndBytes：4-bit NF4 量化，进一步省显存
quant_config = BitsAndBytesConfig(
    # load_in_4bit=True：以 4-bit 权重加载模型
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4"：使用 NormalFloat4 量化类型（常见于 QLoRA 设定）
    bnb_4bit_quant_type="nf4",
    # bnb_4bit_use_double_quant=True：二次量化，再压一点常量开销
    bnb_4bit_use_double_quant=True,
    # bnb_4bit_compute_dtype：反量化后用 bfloat16 做矩阵计算
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. 从 Colab Secrets 取 HF Token；有则 login（add_to_git_credential 按原文保留）
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    # 登录 Hugging Face Hub，便于下载 gated 模型
    login(token=hf_token, add_to_git_credential=True)

# 4. 按 MODEL_ID 加载分词器（Tokenizer）：文本 ↔ token ids
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
# 因果 LM 常无独立 pad_token：把 pad 设成 eos，避免 batch/generate 时报错
tokenizer.pad_token = tokenizer.eos_token

# 5. 加载已量化的因果语言模型到可用设备（device_map="auto" 自动切分到 GPU/CPU）
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=hf_token
)

# 运行时提示字符串保持英文原样
print(f"Model {MODEL_ID} and tokenizer loaded successfully with 4-bit NF4 quantization.")


## 实现生成器逻辑

写一个函数：用已量化的 Llama 模型生成合成记录，再把模型输出解析为 **Pandas DataFrame**，供表格展示或导出。


In [2]:
# ========== 生成器核心：prompt → generate → 抽 JSON → DataFrame ==========

# 导入标准库 json：把模型吐出的 JSON 文本解析成 Python 对象（list/dict）
import json
# 导入标准库 re：用正则从「可能夹杂说明」的回复里抠出 JSON 列表
import re
# 导入 pandas：把 list[dict] 收成表格 DataFrame
import pandas as pd

def generate_synthetic_data(category, num_records):
    """
    Generates synthetic records using the Llama model and parses them into a DataFrame.
    """
    # 1. 根据 HR 策略构建 user prompt（英文原文必须保留：改译会改变模型输出分布）
    prompt = f"""Generate a valid JSON list containing {num_records} synthetic {category} records.
Each record must strictly follow this schema:
- Employee ID: Unique string (e.g., EMP-001)
- Full Name: Realistic name
- Department: Engineering, Sales, Marketing, HR, Finance, or Legal
- Job Title: Appropriate for the department
- Salary: Integer between 40000 and 200000
- Hire Date: YYYY-MM-DD between 2010 and 2024
- Performance Rating: Integer 1-5

Return ONLY the JSON list. Do not include any explanations or markdown code blocks."""

    # 2. 组装 chat messages，再套模型专用 chat template，得到 input_ids 张量
    messages = [{"role": "user", "content": prompt}]
    # apply_chat_template 返回 BatchEncoding；取 input_ids 并搬到模型所在 device
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt")['input_ids'].to(model.device)

    # no_grad：推理不建计算图，省显存、加速
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            # max_new_tokens：最多新生成多少 token（控制长度与耗时）
            max_new_tokens=1000,
            # do_sample=True：采样而非纯贪婪，增加多样性
            do_sample=True,
            # temperature：越高越随机；0.7 是折中
            temperature=0.7,
            # top_p：核采样，只在累计概率质量内采样
            top_p=0.9
        )

    # 3. 把整段 token 序列解码成字符串（skip_special_tokens 去掉特殊符号）
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 4. 用正则抓取「看起来像 JSON 列表」的片段（含数字，降低误匹配空括号）
    try:
        json_match = re.search(r'\[.*\d.*\]', response, re.DOTALL)
        if json_match:
            # group(0)：整段匹配到的 JSON 列表文本
            json_str = json_match.group(0)
            data = json.loads(json_str)
        else:
            # 正则没抓到时：假设整段 response 本身就是合法 JSON
            data = json.loads(response)

        # 5. list/dict → DataFrame，便于 display / 导出
        df = pd.DataFrame(data)
        return df
    except Exception as e:
        # 解析失败时打印错误与原文，返回空表（不中断整本笔记本）
        print(f"Error parsing model output: {e}")
        print("Raw Response:", response)
        return pd.DataFrame()

# 用 3 条 HR/Employee 记录做冒烟测试
test_df = generate_synthetic_data('HR/Employee', 3)
print("Generated Synthetic Data:")
# display：在 Jupyter/Colab 里漂亮展示 DataFrame（需环境已提供 display）
display(test_df)


## 导出到 Google 表格

把上面得到的 `test_df` 挂到 Colab 的 **InteractiveSheet**，方便在表格里查看或继续导出。


In [1]:
# ========== 可选：把 DataFrame 接到 Google Sheets 交互表 ==========

# 从 google.colab 导入 sheets：Colab 提供的表格桥接工具
from google.colab import sheets
# InteractiveSheet：用当前 test_df 打开可交互电子表格视图（参数名/对象保持原样）
sheet = sheets.InteractiveSheet(df=test_df)
